# Stadtbach – PF (Monolith) & RH (Rolling Horizon)

Dieses Notebook ist die **überarbeitete, geprüfte** Fassung:
- **Zelle 1:** Schalter/Umgebungsvariablen setzen (RUN_MODE, PRESET, u.a.)
- **Zelle 2:** Vollständiger Code (PF & RH). Der `__main__`-Auto-Run ist deaktiviert.
- **Zelle 3:** Beispiel-Run (`run_all`) und Ergebnisübersicht.

**Hinweise:**
1. Lege `Import_Data.xlsx` ins gleiche Verzeichnis.
2. Passe die Schalter unten an und führe dann alle Zellen der Reihe nach aus.
3. Exporte landen in `exports/<Zeitstempel>_<PRESET>_<RUN_MODE>/` inkl. `scenario_summary.xlsx`.


In [6]:
# ==============================
# Einheitlicher Konfig-Block (A,C,D refactored)
# ==============================
from __future__ import annotations
import os, re, json, bisect, time, shutil
from pathlib import Path
from typing import Optional, Dict, Any

import numpy as np
import pandas as pd

import pyomo.environ as pyo
from pyomo.environ import (
    ConcreteModel, Var, Param, RangeSet, Constraint, Objective, Expression,
    NonNegativeReals, PositiveReals, Binary, minimize, value as pyo_val
)
from pyomo.opt import SolverFactory

# --- Szenario-Titel früh setzen (optional)
os.environ.setdefault("SCENARIO_TITLE", "HP_v3_CO2_100")

def _getenv(key: str, default, cast=str):
    """ENV lesen, auf Typ casten. Bei leeren Strings auf Default zurückfallen."""
    raw = os.environ.get(key, None)
    val = default if (raw is None or str(raw).strip() == "") else raw
    try:
        return cast(val)
    except Exception:
        return cast(default)

def _getbool(key: str, default="0"):
    return bool(int(_getenv(key, default, str)))

# --- Allgemein / Timing ---
YEAR_TARGET = _getenv("YEAR_TARGET", 2023, int)
DT_H        = _getenv("DT_H", 1.0, float)  # Stunden

# --- Laufmodus & Szenario-Titel ---
RUN_MODE = _getenv("RUN_MODE", "PF_ONLY", str).upper()    # PF_ONLY | RH_ONLY | PF_THEN_RH
assert RUN_MODE in {"PF_ONLY", "RH_ONLY", "PF_THEN_RH"}, f"Ungültiger RUN_MODE: {RUN_MODE}"
SCENARIO_TITLE = _getenv("SCENARIO_TITLE", "Baseline", str)  # eindeutiger, einmaliger Titel

# --- Dateien & Exportpfade ---
EXPORT_BASE_DIR = _getenv("EXPORT_BASE_DIR", "exports", str)
INPUT_XLSX      = _getenv("INPUT_XLSX", "Import_Data.xlsx", str)
INPUT_SHEET     = None  # None → erstes Blatt

# Zentrales Workbook (C)
SCENARIO_XLSX   = str(Path(EXPORT_BASE_DIR) / "scenario.xlsx")

# PF-Design-Export (JSON bleibt für RH_ONLY, C)
PF_DESIGN_JSON = _getenv("PF_DESIGN_JSON", str(Path(EXPORT_BASE_DIR) / "pf_design.json"), str)

# --- Solver ---
SOLVER_NAME = _getenv("SOLVER_NAME", "gurobi", str)  # "gurobi","cbc","glpk"
SOLVER_TEE  = _getbool("SOLVER_TEE", "1")

# --- Rolling Horizon ---
HEAT_HORIZON_HOURS = _getenv("HEAT_HORIZON_HOURS", 7*24, int)
STEP_HOURS         = _getenv("STEP_HOURS", 24, int)
RH_TERMINAL_POLICY = _getenv("RH_TERMINAL_POLICY", "geq", str)  # "equal"|"geq"|"free"

# --- Kosten-Schalter (C: gebündelt) ---
FLAGS = {
    "gridcost_in_energy": _getbool("INCLUDE_GRIDCOST_IN_ENERGY", "0"),
    "demand_in_rh":       _getbool("INCLUDE_DEMAND_CHARGE_IN_RH", "0"),
    "capex_in_rh":        _getbool("INCLUDE_INVEST_COSTS_IN_RH", "0"),
    "install_in_rh":      _getbool("INCLUDE_INSTALL_COSTS_IN_RH", "0"),
    "co2_in_obj":         _getbool("INCLUDE_CO2_COST_IN_OBJECTIVE", "1"),
}

# --- Preise ---
LEISTUNGSPREIS_EUR_PER_MW    = _getenv("LEISTUNGSPREIS_EUR_PER_MW", 127240.0, float)
GRIDCOST_EUR_PER_MWh         = _getenv("GRIDCOST_EUR_PER_MWh", 61.6, float)
GASPREIS_EUR_PER_MWh_th      = _getenv("GASPREIS_EUR_PER_MWh_th", 58.6, float)
BIOMASSEPREIS_EUR_PER_MWh_th = _getenv("BIOMASSEPREIS_EUR_PER_MWh_th", 20.0, float)
ABFALLPREIS_EUR_PER_MWh_th   = _getenv("ABFALLPREIS_EUR_PER_MWh_th", 10.0, float)
CO2_PRICE_EUR_PER_T          = _getenv("CO2_PRICE_EUR_PER_T", 100.0, float)

# --- Einspeise-Mechanik (beibehalten) ---
EINSPEISE_FLOOR_EUR_PER_MWh = _getenv("EINSPEISE_FLOOR_EUR_PER_MWh", 0.0, float)
SELL_HAIRCUT = _getenv("SELL_HAIRCUT", 0.05, float)
SELL_SPREAD  = _getenv("SELL_SPREAD", 5.0, float)
SELL_FEE     = _getenv("SELL_FEE", 5.0, float)
SELL_PREMIUM = _getenv("SELL_PREMIUM", 0.0, float)

# --- Enable-Flags ---
ENABLE_CONFIG = {
    "HP1": True, "HP2": True, "HP3": True, "HP4": True,
    "HKW": True, "GTOST": True, "P2H": True,
    "BMHKW": True, "HWS": True, "HWW": True, "AVA": True,
    "STORAGE": True,
}

def print_config():
    print(f"RUN_MODE       = {RUN_MODE}")
    print(f"SCENARIO_TITLE = {SCENARIO_TITLE}")
    print(f"INPUT          = {INPUT_XLSX}")
    print(f"EXPORTS        = {EXPORT_BASE_DIR}")
print_config()


# ==============================
# 1) Utils & Loader
# ==============================
def _slug(s: str, maxlen: int = 60) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", "-", s)
    s = re.sub(r"[^A-Za-z0-9._-]+", "", s)
    return (s or "Baseline")[:maxlen]

def _require(condition: bool, msg: str):
    if not condition:
        raise RuntimeError(msg)

def _excel_safe_df(df: pd.DataFrame, tz: str = "Europe/Berlin") -> pd.DataFrame:
    """Entfernt Zeitzonen für Excel-Writer (Index & datetime-Spalten)."""
    df2 = df.copy()
    if isinstance(df2.index, pd.DatetimeIndex) and df2.index.tz is not None:
        df2.index = df2.index.tz_convert(tz).tz_localize(None)
    for c in df2.columns:
        if np.issubdtype(df2[c].dtype, np.datetime64):
            s = pd.to_datetime(df2[c], errors="coerce")
            try:
                if getattr(s.dt, "tz", None) is not None:
                    s = s.dt.tz_convert(tz).dt.tz_localize(None)
            except Exception:
                pass
            df2[c] = s
    return df2

_norm = lambda s: re.sub(r"[^a-z0-9]+", "", str(s).lower())
def _get_col(df: pd.DataFrame, *candidates: str) -> str:
    norm_map = {_norm(c): c for c in df.columns}
    for cand in candidates:
        key = _norm(cand)
        if key in norm_map:
            return norm_map[key]
    for cand in candidates:
        key = _norm(cand)
        for k, orig in norm_map.items():
            if k == key or k.startswith(key) or key in k:
                return orig
    raise RuntimeError("Fehlende Spalte. Gesucht eine von: " + " | ".join(candidates))

def _parse_and_localize_datetime(
    sr: pd.Series,
    tz: str = "Europe/Berlin",
    input_time_is_local: bool = True,
    ambiguous_policy: str = "infer_then_first"  # "infer_then_first" | "first" | "last" | "NaT" | "infer"
) -> pd.DatetimeIndex:
    ts = pd.to_datetime(sr, errors="raise", utc=False)
    if getattr(ts.dt, "tz", None) is None:
        if input_time_is_local:
            if ambiguous_policy == "infer_then_first":
                try:
                    ts = ts.dt.tz_localize(tz, nonexistent="shift_forward", ambiguous="infer")
                except Exception:
                    ts = ts.dt.tz_localize(tz, nonexistent="shift_forward", ambiguous=True)
            else:
                ambiguous_arg = {"first": True, "last": False, "NaT": "NaT", "infer": "infer"}.get(ambiguous_policy, "infer")
                ts = ts.dt.tz_localize(tz, nonexistent="shift_forward", ambiguous=ambiguous_arg)
        else:
            ts = ts.dt.tz_localize("UTC")
    ts_utc_naive = ts.dt.tz_convert("UTC").dt.tz_localize(None)
    return pd.DatetimeIndex(ts_utc_naive)

def _enforce_monotonic_unique_index(
    df: pd.DataFrame,
    duplicate_strategy: str = "drop_first"
) -> pd.DataFrame:
    if not isinstance(df.index, pd.DatetimeIndex):
        raise RuntimeError("Index ist kein Datumsindex.")
    df = df.sort_index()
    dup_mask = df.index.duplicated(keep="first")
    n_dup = int(dup_mask.sum())
    if n_dup:
        if duplicate_strategy == "drop_first":
            print(f"[Loader] Entferne {n_dup} doppelte Zeitstempel (behalte erste Vorkommen).")
            df = df[~dup_mask]
        else:
            raise RuntimeError(f"Doppelte Zeitstempel gefunden: {n_dup}.")
    _require(df.index.is_monotonic_increasing, "Zeitindex nicht monoton steigend.")
    _require(df.index.is_unique, "Zeitindex enthält weiterhin Duplikate.")
    return df

def _reindex_hourly_and_fill(
    df: pd.DataFrame,
    dt_hours: float = 1.0,
    gap_strategy: str = "error"
) -> pd.DataFrame:
    if df.empty:
        return df
    freq = pd.to_timedelta(dt_hours, unit="h")
    target_index = pd.date_range(df.index.min(), df.index.max(), freq=freq)
    if len(target_index) != len(df):
        missing = len(target_index) - len(df)
        msg = f"[Loader] Es fehlen {missing} Zeitschritte (soll: {len(target_index)}, ist: {len(df)})."
        if gap_strategy == "error":
            raise RuntimeError(msg + " gap_strategy='error' → abbrechen.")
        print(msg + f" gap_strategy='{gap_strategy}' → fülle Lücken.")
    df2 = df.reindex(target_index)
    if gap_strategy == "ffill":
        df2 = df2.ffill().bfill()
    elif gap_strategy == "bfill":
        df2 = df2.bfill().ffill()
    elif gap_strategy == "interp":
        num_cols = df2.select_dtypes(include=[np.number]).columns
        non_num  = [c for c in df2.columns if c not in num_cols]
        df2[num_cols] = df2[num_cols].interpolate(method="time", limit_direction="both")
        if non_num:
            df2[non_num] = df2[non_num].ffill().bfill()
    steps = df2.index.to_series().diff().dropna().unique()
    _require(len(steps) == 1 and steps[0] == freq, f"Schrittweite ist nicht {freq}.")
    return df2

def load_input_excel(
    path: str,
    sheet_name=None,
    year_target: int | None = None,
    *,
    tz: str = "Europe/Berlin",
    input_time_is_local: bool = True,
    dt_hours: float | None = None,
    duplicate_strategy: str = "drop_first",
    gap_strategy: str = "error",
    ambiguous_policy: str = "infer_then_first",
) -> pd.DataFrame:
    _require(os.path.exists(path), f"Datei nicht gefunden: {path}")
    xls = pd.ExcelFile(path)
    sh  = sheet_name if sheet_name is not None else xls.sheet_names[0]
    df_raw = pd.read_excel(path, sheet_name=sh)
    df_raw.columns = [str(c).strip() for c in df_raw.columns]

    _require("Datum" in df_raw.columns, "Spalte 'Datum' fehlt in Import_Data.xlsx")
    idx = _parse_and_localize_datetime(
        df_raw["Datum"],
        tz=tz,
        input_time_is_local=input_time_is_local,
        ambiguous_policy=ambiguous_policy,
    )

    df = df_raw.set_index(idx).drop(columns=["Datum"])
    if year_target is not None:
        df = df[df.index.year == int(year_target)]
        _require(len(df) > 0, f"Im Zieljahr {year_target} keine Daten.")
    df = _enforce_monotonic_unique_index(df, duplicate_strategy=duplicate_strategy)

    col_price = _get_col(df, "Day_Ahead_Price €/MWh", "Day Ahead Price €/MWh", "strompreis", "price_eur_mwh")
    col_heat  = _get_col(df, "Wärmebedarf MW", "Waermebedarf MW", "waermebedarf", "heat_demand_mw")
    col_co2   = _get_col(df, "CO2_consumption_based kgCO2/MWh", "co2_intensity_kgco2_mwh", "co2 kgco2/mwh")

    col_wrgq = {i: _get_col(df,
                            f"WRG{i}Q MW", f"WRG{i} Q MW", f"WRG{i}_Q MW", f"WRG{i}_Q MW_th", f"wrg{i}qmw")
                for i in [1,2,3,4]}
    col_wrgt = {i: _get_col(df,
                            f"WRG{i}_T °C", f"WRG{i} T °C", f"wrg{i} temp c")
                for i in [1,2,3,4]}

    out = pd.DataFrame(index=df.index)
    _to_num = lambda s: pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="raise")

    out["strompreis_EUR_MWh"] = _to_num(df[col_price])
    out["waermebedarf_MWth"]  = _to_num(df[col_heat])
    out["grid_co2_kg_MWh"]    = _to_num(df[col_co2])

    for i in [1,2,3,4]:
        out[f"WRG{i}_Q_cap"] = _to_num(df[col_wrgq[i]])
        out[f"WRG{i}_T_K"]   = _to_num(df[col_wrgt[i]]) + 273.15

    _require(not out.isna().any().any(), "NaN in Import_Data – bitte fehlende Werte schließen.")

    if dt_hours is None:
        dt_hours = float(globals().get("DT_H", 1.0))
    out = _reindex_hourly_and_fill(out, dt_hours=dt_hours, gap_strategy=gap_strategy)

    print(f"[LOAD] {os.path.basename(path)} → {len(out)} Std (UTC-naiv) "
          f"von {out.index.min()} bis {out.index.max()}, dt={dt_hours}h")
    return out


# ==============================
# 2) COP-Logik (wie gehabt) – D bleibt
# ==============================
Tsink_out = 363.15; Tsink_in = 343.15
deltaTpp  = 5.0
eta       = 0.75
FQ        = 0.10

def lmtd(Th, Tc):
    return (Th - Tc) / np.log(Th / Tc)

LMTD_sink = lmtd(Tsink_out, Tsink_in)

Tsourcein_vals = np.linspace(303.15, 353.15, 6)
deltaT_vals    = np.array([10, 20, 30, 40, 50])
_records = []
for Tsourcein in Tsourcein_vals:
    for dT in deltaT_vals:
        Tsourceout = Tsourcein - dT
        if Tsourceout <= 0 or Tsourceout >= Tsourcein:
            continue
        LMTD_source = lmtd(Tsourcein, Tsourceout)
        mdts = 0.2*(Tsink_out - Tsourceout + 2*deltaTpp) + 0.2*(Tsink_out - Tsink_in) + 0.016
        qww  = 0.0014*(Tsink_out - Tsourceout + 2*deltaTpp) - 0.0015*(Tsink_out - Tsink_in) + 0.039
        A = LMTD_sink / (LMTD_sink - LMTD_source + 1e-9)
        B = (1 + (mdts + deltaTpp)/LMTD_sink) / (1 + (mdts + 0.5*dT + 2*deltaTpp)/(LMTD_sink - LMTD_source + 1e-9))
        COP = A * B * eta * (1 - qww) + 1 - eta - FQ
        COP = float(np.clip(COP if np.isfinite(COP) else 3.0, 0.5, 12.0))
        _records.append({"Tsourcein": round(Tsourcein,2), "Tsourceout": round(Tsourceout,2), "COP": round(COP,4)})
cop_lookup_df = pd.DataFrame(_records).sort_values(["Tsourcein","Tsourceout"])

from collections import defaultdict
cop_piecewise_rules = defaultdict(list)
for _, r in cop_lookup_df.iterrows():
    cop_piecewise_rules[r["Tsourcein"]].append((r["Tsourceout"], r["COP"]))
cop_piecewise_rules = {k: sorted(v, key=lambda x: x[0]) for k, v in cop_piecewise_rules.items()}
_all_x = sorted({x for pts in cop_piecewise_rules.values() for (x, _) in pts})
x_min, x_max = (_all_x[0], _all_x[-1]) if _all_x else (0.0, 1.0)
T_grid = sorted(cop_piecewise_rules.keys())

def _interp_on_curve(pts, x):
    if x <= pts[0][0]:  return pts[0][1]
    if x >= pts[-1][0]: return pts[-1][1]
    for (x0,y0),(x1,y1) in zip(pts[:-1], pts[1:]):
        if x0 <= x <= x1:
            return y0 + (y1-y0) * (x - x0) / (x1 - x0)
    return pts[-1][1]

def bilinear_interp_from_lookup(Tsrc_in, x, clamp_x=True):
    if not T_grid: return 3.0
    if clamp_x: x = max(x_min, min(x_max, x))
    j = bisect.bisect_left(T_grid, Tsrc_in)
    if j == 0: return _interp_on_curve(cop_piecewise_rules[T_grid[0]], x)
    if j == len(T_grid): return _interp_on_curve(cop_piecewise_rules[T_grid[-1]], x)
    t0, t1 = T_grid[j-1], T_grid[j]
    y0 = _interp_on_curve(cop_piecewise_rules[t0], x)
    y1 = _interp_on_curve(cop_piecewise_rules[t1], x)
    y0 = 3.0 if (not np.isfinite(y0)) else y0
    y1 = 3.0 if (not np.isfinite(y1)) else y1
    w  = (Tsrc_in - t0) / (t1 - t0)
    val = y0*(1-w) + y1*w
    return float(np.clip(val if np.isfinite(val) else 3.0, 0.5, 12.0))

def safe_cop(Tin, Tout, COP_MIN=1.01, COP_MAX=12.0, COP_FALLBACK=3.0):
    Tin_v, Tout_v = float(Tin), float(Tout)
    x = max(x_min, min(x_max, float(Tout_v))) if x_min < x_max else float(Tout_v)
    val = bilinear_interp_from_lookup(Tin_v, x) if T_grid else COP_FALLBACK
    if (not np.isfinite(val)) or (val <= 0):
        raise RuntimeError(f"COP-Berechnung fehlgeschlagen (Tin={Tin}, Tout={Tout}).")
    return float(np.clip(val, COP_MIN, COP_MAX))


# ==============================
# 3) Modell-Builder (PF & RH) – (A) indexierte HPs, (D) Terminal-Policy vereinfacht
# ==============================
HP_IDS = [1,2,3,4]

def _apply_terminal_policy(m: pyo.ConcreteModel, policy: str):
    """Einheitliche Umsetzung: >=, optional <=, oder frei (D)."""
    # Basis: >=
    m.soc_terminal_geq = pyo.Constraint(expr = m.storage_level[m.t.last()] >= m.SOC_init)
    if policy == "equal":
        m.soc_terminal_eq = pyo.Constraint(expr = m.storage_level[m.t.last()] <= m.SOC_init)
    elif policy == "geq":
        pass
    elif policy == "free":
        m.soc_terminal_geq.deactivate()
    else:
        raise ValueError(f"Unbekannte RH_TERMINAL_POLICY: {policy}")

def _build_common_blocks(m: pyo.ConcreteModel, T: int, dfW: pd.DataFrame):
    # Grundreihen
    prices = dfW["strompreis_EUR_MWh"].astype(float).tolist()
    heat   = dfW["waermebedarf_MWth"].astype(float).tolist()
    co2    = dfW["grid_co2_kg_MWh"].astype(float).tolist()

    def _cop_series(T_K: pd.Series, dT=20.0):
        Tin = T_K.astype(float).values
        Tout = np.maximum(1.0, Tin - dT)
        return [safe_cop(Tin[i], Tout[i]) for i in range(T)]

    # Mengen
    m.t = RangeSet(1, T)
    m.HP = pyo.Set(initialize=HP_IDS)

    # Parameter
    m.strompreis   = Param(m.t, initialize={i: prices[i-1] for i in range(1, T+1)})
    m.waermebedarf = Param(m.t, initialize={i: heat[i-1]   for i in range(1, T+1)})
    m.grid_co2_kg_per_MWh = Param(m.t, initialize={i: co2[i-1] for i in range(1, T+1)}, within=NonNegativeReals, mutable=True)

    # Kosten-Parameter
    m.Leistungspreis      = Param(initialize=LEISTUNGSPREIS_EUR_PER_MW, mutable=True)
    m.Gridcost            = Param(initialize=GRIDCOST_EUR_PER_MWh, mutable=True)
    m.Installationskosten = Param(initialize=20000.0, mutable=True)
    m.Gaspreis            = Param(initialize=GASPREIS_EUR_PER_MWh_th, mutable=True)
    m.Abfallpreis         = Param(initialize=ABFALLPREIS_EUR_PER_MWh_th, mutable=True)
    m.Biomassepreis       = Param(initialize=BIOMASSEPREIS_EUR_PER_MWh_th, mutable=True)

    # Einspeise-Mechanik
    m.einspeisepreis = Param(initialize=EINSPEISE_FLOOR_EUR_PER_MWh, within=NonNegativeReals, mutable=True)
    m.SELL_HAIRCUT = Param(initialize=SELL_HAIRCUT,  mutable=True)
    m.SELL_SPREAD  = Param(initialize=SELL_SPREAD,   mutable=True)
    m.SELL_FEE     = Param(initialize=SELL_FEE,      mutable=True)
    m.SELL_PREMIUM = Param(initialize=SELL_PREMIUM,  mutable=True)
    m.sell_base = Expression(m.t, rule=lambda mm, t: (1.0 - mm.SELL_HAIRCUT) * mm.strompreis[t]
                                               + mm.SELL_PREMIUM - mm.SELL_FEE - mm.SELL_SPREAD)
    m.sell_price_eff = Param(m.t, initialize=lambda mm, t: max(float(pyo_val(mm.sell_base[t])),
                                                               float(pyo_val(mm.einspeisepreis))), mutable=True)

    # CO2
    m.CO2_price_EUR_per_t = Param(initialize=float(CO2_PRICE_EUR_PER_T), mutable=True)
    m.CO2_cost_switch     = Param(initialize=int(bool(FLAGS["co2_in_obj"])), mutable=True, within=NonNegativeReals)

    # Enable-Flags (indexiert für HPs)
    m.EN_HP = Param(m.HP, initialize={i: int(ENABLE_CONFIG[f"HP{i}"]) for i in HP_IDS},
                    within=NonNegativeReals, mutable=True)
    m.EN_HKW     = Param(initialize=int(ENABLE_CONFIG["HKW"]),     within=NonNegativeReals, mutable=True)
    m.EN_GTOST   = Param(initialize=int(ENABLE_CONFIG["GTOST"]),   within=NonNegativeReals, mutable=True)
    m.EN_P2H     = Param(initialize=int(ENABLE_CONFIG["P2H"]),     within=NonNegativeReals, mutable=True)
    m.EN_BMHKW   = Param(initialize=int(ENABLE_CONFIG["BMHKW"]),   within=NonNegativeReals, mutable=True)
    m.EN_HWS     = Param(initialize=int(ENABLE_CONFIG["HWS"]),     within=NonNegativeReals, mutable=True)
    m.EN_HWW     = Param(initialize=int(ENABLE_CONFIG["HWW"]),     within=NonNegativeReals, mutable=True)
    m.EN_AVA     = Param(initialize=int(ENABLE_CONFIG["AVA"]),     within=NonNegativeReals, mutable=True)
    m.EN_STORAGE = Param(initialize=int(ENABLE_CONFIG["STORAGE"]), within=NonNegativeReals, mutable=True)

    # Speicher
    m.storage_eff_charge    = Param(initialize=0.95)
    m.storage_eff_discharge = Param(initialize=0.95)
    m.storage_c_rate        = Param(initialize=0.25)
    m.CAPEXspeicher       = Param(initialize=5160.0, within=PositiveReals, mutable=True)
    m.Lebensdauerspeicher = Param(initialize=20)
    STO_E_MAX = 50000.0; STO_E_MIN = 10.0; STO_P_MAX = 50.0
    m.storage_capacity        = Var(bounds=(0, STO_E_MAX))
    m.storage_power           = Var(bounds=(0, STO_P_MAX))
    m.storage_capacity_active = Var(domain=Binary)
    m.storage_level     = Var(m.t, domain=NonNegativeReals)
    m.storage_charge    = Var(m.t, domain=NonNegativeReals)
    m.storage_discharge = Var(m.t, domain=NonNegativeReals)
    m.SOC_init          = Param(initialize=0.0, within=NonNegativeReals, mutable=True)
    m.sto_mode          = Var(m.t, domain=Binary)
    m.sto_mode_gate = pyo.Constraint(m.t, rule=lambda mm,t: mm.sto_mode[t] <= mm.storage_capacity_active)
    m.storage_power_rate = pyo.Constraint(expr=m.storage_power <= m.storage_c_rate * m.storage_capacity)
    m.sto_gate_charge    = pyo.Constraint(m.t, rule=lambda mm, t: mm.storage_charge[t]    <= mm.storage_power * mm.EN_STORAGE * mm.sto_mode[t])
    m.sto_gate_discharge = pyo.Constraint(m.t, rule=lambda mm, t: mm.storage_discharge[t] <= mm.storage_power * mm.EN_STORAGE * (1 - mm.sto_mode[t]))
    m.soc_cap            = pyo.Constraint(m.t, rule=lambda mm, t: mm.storage_level[t]     <= mm.storage_capacity * mm.EN_STORAGE)
    m.storage_linkE_hi   = pyo.Constraint(expr=m.storage_capacity <= STO_E_MAX * m.storage_capacity_active * m.EN_STORAGE)
    m.storage_linkE_lo   = pyo.Constraint(expr=m.storage_capacity >= STO_E_MIN * m.storage_capacity_active * m.EN_STORAGE)
    m.storage_active_gate= pyo.Constraint(expr=m.storage_capacity_active <= m.EN_STORAGE)
    m.storage_loss_hour  = Param(initialize=0.9999, mutable=True)
    m.storage_loss_step  = pyo.Expression(rule=lambda mm: mm.storage_loss_hour ** DT_H)
    def storage_state_rule(mm, t):
        prev = (mm.storage_level[t-1] * mm.storage_loss_step) if t > mm.t.first() else mm.SOC_init
        return mm.storage_level[t] == prev \
            + mm.storage_eff_charge    * mm.storage_charge[t]    * DT_H \
            - (1.0 / mm.storage_eff_discharge) * mm.storage_discharge[t] * DT_H
    m.storage_state   = pyo.Constraint(m.t, rule=storage_state_rule)
    # Terminal-Policy wird später per _apply_terminal_policy() gesetzt

    # Konventionelle Aggregate
    m.HKW_th_eff   = Param(initialize=0.743)
    m.HKW_el_eff   = Param(initialize=0.177)
    m.GTOST_th_eff = Param(initialize=0.466)
    m.GTOST_el_eff = Param(initialize=0.36)
    m.P2H_eff      = Param(initialize=0.99)
    m.BMHKW_th_eff = Param(initialize=0.485)
    m.BMHKW_el_eff = Param(initialize=0.177)
    m.HWS_th_eff   = Param(initialize=0.936)
    m.HWW_th_eff   = Param(initialize=0.924)
    m.AVA_th_eff   = Param(initialize=1.0)

    m.HKW_Power   = Var(m.t, domain=NonNegativeReals, bounds=(0, 75))
    m.GTOST_Power = Var(m.t, domain=NonNegativeReals, bounds=(0, 41.3))
    m.P2H_Power   = Var(m.t, domain=NonNegativeReals, bounds=(0, 10))
    m.BMHKW_Power = Var(m.t, domain=NonNegativeReals, bounds=(0, 15))
    m.HWS_Power   = Var(m.t, domain=NonNegativeReals, bounds=(0, 45))
    m.HWW_Power   = Var(m.t, domain=NonNegativeReals, bounds=(0, 45))
    m.AVA_Power   = Var(m.t, domain=NonNegativeReals, bounds=(0, 45))

    # Wärmepumpen – indexierte Designvariablen (A)
    HP_MAX = 100.0
    m.HPNenn        = Var(m.HP, bounds=(0, HP_MAX))
    m.HPNenn_active = Var(m.HP, domain=Binary)
    m.onHP          = Var(m.HP, m.t, domain=Binary)
    m.HP_min_load_fraction = Param(initialize=0.30)

    # WRG-Caps & COP (indexiert)
    WRGQ = {i: dfW.get(f"WRG{i}_Q_cap", pd.Series([0]*T, index=dfW.index)).astype(float).tolist() for i in HP_IDS}
    COPs = {i: (_cop_series(dfW[f"WRG{i}_T_K"]) if f"WRG{i}_T_K" in dfW.columns else [3.0]*T) for i in HP_IDS}
    m.WRGQ = Param(m.HP, m.t, initialize={(i,t): WRGQ[i][t-1] for i in HP_IDS for t in range(1,T+1)},
                   within=NonNegativeReals, mutable=True)
    m.COP  = Param(m.HP, m.t, initialize={(i,t): COPs[i][t-1] for i in HP_IDS for t in range(1,T+1)},
                   within=PositiveReals)
    m.COPdefault = Param(initialize=3.0, within=PositiveReals)

    m.Qwrg = Var(m.HP, m.t, domain=NonNegativeReals)
    m.Qdef = Var(m.HP, m.t, domain=NonNegativeReals)
    m.Qhp  = Var(m.HP, m.t, domain=NonNegativeReals)

    # WRG Caps & HP-Bilanz
    m.wrg_cap = pyo.ConstraintList()
    m.hp_caps = pyo.ConstraintList()
    m.on_impl = pyo.ConstraintList()
    m.hp_link_hi = pyo.ConstraintList()
    m.hp_link_lo = pyo.ConstraintList()
    m.hp_enable_on = pyo.ConstraintList()

    for i in HP_IDS:
        for t in range(1, T+1):
            m.wrg_cap.add(m.Qwrg[i,t] <= m.WRGQ[i,t])
            m.hp_caps.add(m.Qhp[i,t] == m.Qwrg[i,t] + m.Qdef[i,t])
            m.hp_caps.add(m.Qhp[i,t] <= m.HPNenn[i])
            m.hp_caps.add(m.Qhp[i,t] <= 100.0 * m.onHP[i,t])
            m.hp_caps.add(m.Qhp[i,t] >= m.HP_min_load_fraction * m.HPNenn[i]
                          - (1 - m.onHP[i,t]) * m.HP_min_load_fraction * 100.0)
            m.on_impl.add(m.onHP[i,t] <= m.HPNenn_active[i])
            m.hp_enable_on.add(m.onHP[i,t] <= m.EN_HP[i])
        m.hp_link_hi.add(m.HPNenn[i] <= 100.0 * m.HPNenn_active[i] * m.EN_HP[i])
        m.hp_link_lo.add(m.HPNenn[i] >= 1.0   * m.HPNenn_active[i] * m.EN_HP[i])

    # Wärmebilanz + Dump (indexierte Summe)
    m.total_heat = pyo.Expression(m.t, rule=lambda mm, t: (
        pyo.quicksum(mm.Qhp[i,t] for i in mm.HP)
      + mm.HKW_Power[t]   * mm.HKW_th_eff   * mm.EN_HKW
      + mm.GTOST_Power[t] * mm.GTOST_th_eff * mm.EN_GTOST
      + mm.P2H_Power[t]   * mm.P2H_eff      * mm.EN_P2H
      + mm.BMHKW_Power[t] * mm.BMHKW_th_eff * mm.EN_BMHKW
      + mm.HWS_Power[t]   * mm.HWS_th_eff   * mm.EN_HWS
      + mm.HWW_Power[t]   * mm.HWW_th_eff   * mm.EN_HWW
      + mm.AVA_Power[t]   * mm.AVA_th_eff   * mm.EN_AVA
    ))
    m.Q_dump = Var(m.t, domain=NonNegativeReals)
    m.waerme_balance = pyo.Constraint(m.t, rule=lambda mm, t:
        mm.total_heat[t] - mm.storage_charge[t] + mm.storage_discharge[t] ==
        mm.waermebedarf[t] + mm.Q_dump[t]
    )

    # Strombilanz
    P_FLOW_CAP = 300.0
    m.P_buy  = Var(m.t, domain=NonNegativeReals, bounds=(0, P_FLOW_CAP))
    m.P_sell = Var(m.t, domain=NonNegativeReals, bounds=(0, P_FLOW_CAP))
    m.e_balance = pyo.Constraint(m.t, rule=lambda mm, t:
        (mm.HKW_Power[t]*mm.HKW_el_eff     * mm.EN_HKW
       + mm.GTOST_Power[t]*mm.GTOST_el_eff * mm.EN_GTOST
       + mm.BMHKW_Power[t]*mm.BMHKW_el_eff * mm.EN_BMHKW
       + mm.P_buy[t])
       - ( pyo.quicksum(mm.Qwrg[i,t]/mm.COP[i,t] + mm.Qdef[i,t]/mm.COPdefault for i in mm.HP)
         + mm.P2H_Power[t] * mm.EN_P2H
         + mm.P_sell[t]) == 0
    )

    # Demand Charge & Grid-Kosten
    m.max_stromverbrauch = Var(domain=NonNegativeReals)
    m.max_stromverbrauch_constraint = pyo.Constraint(m.t, rule=lambda mm, t: mm.max_stromverbrauch >= mm.P_buy[t])
    if FLAGS["gridcost_in_energy"]:
        m.buy_price = pyo.Expression(m.t, rule=lambda mm,t: mm.strompreis[t] + mm.Gridcost)
    else:
        m.buy_price = pyo.Expression(m.t, rule=lambda mm,t: mm.strompreis[t])

    m.M_GRID    = Param(initialize=P_FLOW_CAP)
    m.grid_mode = Var(m.t, domain=Binary)
    m.buy_gate  = pyo.Constraint(m.t, rule=lambda mm, t: mm.P_buy[t]  <= mm.M_GRID * mm.grid_mode[t])
    m.sell_gate = pyo.Constraint(m.t, rule=lambda mm, t: mm.P_sell[t] <= mm.M_GRID * (1 - mm.grid_mode[t]))

    HOURS_PER_YEAR = 8760.0
    m.year_frac = Param(initialize=float(T) * DT_H / HOURS_PER_YEAR)
    m.EPS_REG = Param(initialize=1e-6, mutable=True)

    # Tie-break (D): nur aktiv, wenn KEIN CAPEX in der Zielfunktion aktiv ist.
    m.hp_tiebreak_sum = pyo.Expression(rule=lambda mm: pyo.quicksum(mm.HPNenn[i] for i in mm.HP))

    # CO2 Summen
    m.EF_HKW_kg_per_MWh_fuel     = Param(initialize=201.6, mutable=True)
    m.EF_GTOST_kg_per_MWh_fuel   = Param(initialize=201.6, mutable=True)
    m.EF_HWS_kg_per_MWh_fuel     = Param(initialize=201.6, mutable=True)
    m.EF_HWW_kg_per_MWh_fuel     = Param(initialize=201.6, mutable=True)
    m.EF_BMHKW_kg_per_MWh_fuel   = Param(initialize=0.0,   mutable=True)
    m.EF_AVA_kg_per_MWh_fuel     = Param(initialize=0.0,   mutable=True)

    m.CO2_grid_kg = pyo.Expression(m.t, rule=lambda mm, t: mm.P_buy[t] * mm.grid_co2_kg_per_MWh[t])
    m.CO2_total_kg = pyo.Expression(
        m.t, rule=lambda mm, t:
            mm.CO2_grid_kg[t]
          + mm.HKW_Power[t]   * mm.EF_HKW_kg_per_MWh_fuel   * mm.EN_HKW
          + mm.GTOST_Power[t] * mm.EF_GTOST_kg_per_MWh_fuel * mm.EN_GTOST
          + mm.HWS_Power[t]   * mm.EF_HWS_kg_per_MWh_fuel   * mm.EN_HWS
          + mm.HWW_Power[t]   * mm.EF_HWW_kg_per_MWh_fuel   * mm.EN_HWW
          + mm.BMHKW_Power[t] * mm.EF_BMHKW_kg_per_MWh_fuel * mm.EN_BMHKW
          + mm.AVA_Power[t]   * mm.EF_AVA_kg_per_MWh_fuel   * mm.EN_AVA
    )
    m.CO2_total_tons = pyo.Expression(rule=lambda mm: pyo.quicksum(DT_H * mm.CO2_total_kg[t] for t in mm.t) / 1000.0)
    m.CO2_cost_EUR   = pyo.Expression(rule=lambda mm: mm.CO2_cost_switch * mm.CO2_price_EUR_per_t * mm.CO2_total_tons)

def _objective_common(m: pyo.ConcreteModel, include_demand_charge=True, include_capex=True, include_install=True):
    """Zielfunktion: Opex, optional DemandCharge, CAPEX/Install, CO2, Dump & ggf. Tiebreak (D)."""
    DUMP_COST = 1.0  # €/MWh_th
    oper_cost = (
        pyo.quicksum(DT_H * m.P_buy[t]  * m.buy_price[t]      for t in m.t)
      - pyo.quicksum(DT_H * m.P_sell[t] * m.sell_price_eff[t] for t in m.t)
      + pyo.quicksum(DT_H * (m.HKW_Power[t]   * m.Gaspreis * m.EN_HKW
                           + m.GTOST_Power[t] * m.Gaspreis * m.EN_GTOST
                           + m.HWS_Power[t]   * m.Gaspreis * m.EN_HWS
                           + m.HWW_Power[t]   * m.Gaspreis * m.EN_HWW) for t in m.t)
      + pyo.quicksum(DT_H * (m.AVA_Power[t]   * m.Abfallpreis   * m.EN_AVA)    for t in m.t)
      + pyo.quicksum(DT_H * (m.BMHKW_Power[t] * m.Biomassepreis * m.EN_BMHKW)  for t in m.t)
      + pyo.quicksum(DT_H * DUMP_COST * m.Q_dump[t] for t in m.t)
      + m.CO2_cost_EUR
    )

    demand = (m.Leistungspreis * m.max_stromverbrauch * m.year_frac) if include_demand_charge else 0.0
    capex_hp  = ((600000.0 / 15.0) * pyo.quicksum(m.HPNenn[i] for i in m.HP) * m.year_frac) if include_capex else 0.0
    capex_sto = ((m.CAPEXspeicher / m.Lebensdauerspeicher) * m.storage_capacity * m.year_frac) if include_capex else 0.0
    install_hp = ((m.Installationskosten / 15.0) * pyo.quicksum(m.HPNenn_active[i] for i in m.HP) * m.year_frac) if include_install else 0.0
    install_st = ((m.Installationskosten / m.Lebensdauerspeicher) * m.storage_capacity_active * m.year_frac) if include_install else 0.0

    # Tie-break Gewicht (D): nur wenn keine CAPEX-Kosten
    tiebreak = (m.EPS_REG * m.hp_tiebreak_sum) if (not include_capex) else 0.0

    return oper_cost + demand + capex_hp + capex_sto + install_hp + install_st + tiebreak

def build_pf_model(df_full: pd.DataFrame) -> pyo.ConcreteModel:
    T = len(df_full)
    m = ConcreteModel(name="Stadtbach_PF")
    _build_common_blocks(m, T, df_full)
    include_demand = True
    include_capex  = True
    include_install= True
    m.OBJ = Objective(expr=_objective_common(m, include_demand, include_capex, include_install), sense=minimize)
    # Terminal-Policy ist für PF irrelevant (monolithisch), aber wir setzen >= zur Sicherheit:
    _apply_terminal_policy(m, "geq")
    return m

def build_rh_model(df_win: pd.DataFrame, fixed_design: dict | None = None) -> pyo.ConcreteModel:
    T = len(df_win)
    m = ConcreteModel(name="Stadtbach_RH")
    _build_common_blocks(m, T, df_win)

    include_demand = FLAGS["demand_in_rh"]
    include_capex  = FLAGS["capex_in_rh"]
    include_install= FLAGS["install_in_rh"]
    m.OBJ = Objective(expr=_objective_common(m, include_demand, include_capex, include_install), sense=minimize)

    # PF-Design fixieren (A kompatibel – mappt alte JSON-Keys)
    if fixed_design is not None:
        # HPNenn[i]
        for i in HP_IDS:
            key = f"HPNenn{i}"
            if key in fixed_design:
                m.HPNenn[i].fix(float(fixed_design[key]))
        for i in HP_IDS:
            key = f"HPNenn_active{i}"
            if key in fixed_design:
                m.HPNenn_active[i].fix(int(fixed_design[key]))
        if "storage_capacity" in fixed_design:
            m.storage_capacity.fix(float(fixed_design["storage_capacity"]))
        if "storage_power" in fixed_design:
            m.storage_power.fix(float(fixed_design["storage_power"]))
        if "storage_capacity_active" in fixed_design:
            m.storage_capacity_active.fix(int(fixed_design["storage_capacity_active"]))

    return m


# ==============================
# 4) Solver-Routinen & Exporte – (C) einheitliches Workbook, (A) indexierte Exporte
# ==============================
def _extract_pf_design(m) -> dict:
    d = {
        "storage_capacity": float(pyo_val(m.storage_capacity)),
        "storage_power": float(pyo_val(m.storage_power)),
        "storage_capacity_active": int(round(pyo_val(m.storage_capacity_active))),
    }
    for i in HP_IDS:
        d[f"HPNenn{i}"] = float(pyo_val(m.HPNenn[i]))
        d[f"HPNenn_active{i}"] = int(round(pyo_val(m.HPNenn_active[i])))
    return d

def _timeseries_from_model(m: pyo.ConcreteModel, idx: pd.DatetimeIndex) -> pd.DataFrame:
    ts = pd.DataFrame(index=idx)
    # Grid
    ts["P_buy"]  = [pyo_val(m.P_buy[t]) for t in m.t]
    ts["P_sell"] = [pyo_val(m.P_sell[t]) for t in m.t]
    # HPs (breit + Summe)
    for i in HP_IDS:
        ts[f"HP_Q_{i}"] = [pyo_val(m.Qhp[i,t]) for t in m.t]
    ts["HP_Q_total"] = ts[[f"HP_Q_{i}" for i in HP_IDS]].sum(axis=1)
    # Anlagen
    for name, comp in {
        "HKW_P": m.HKW_Power, "GTOST_P": m.GTOST_Power, "P2H_P": m.P2H_Power,
        "BMHKW_P": m.BMHKW_Power, "HWS_P": m.HWS_Power, "HWW_P": m.HWW_Power, "AVA_P": m.AVA_Power,
        "STO_level": m.storage_level, "STO_charge": m.storage_charge, "STO_discharge": m.storage_discharge,
        "Q_dump": m.Q_dump
    }.items():
        ts[name] = [pyo_val(comp[t]) for t in m.t]
    return ts

def _summary_costs(m: pyo.ConcreteModel, ts: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "OBJ_value_EUR": float(pyo_val(m.OBJ)),
        "P_buy_peak_MW": float(ts["P_buy"].max(skipna=True)),
        "P_buy_sum_MWh": float((ts["P_buy"] * DT_H).sum(skipna=True)),
        "P_sell_sum_MWh": float((ts["P_sell"] * DT_H).sum(skipna=True)),
        "HP_total_heat_MWh": float((ts["HP_Q_total"] * DT_H).sum(skipna=True)),
        "CO2_total_tons": float(pyo_val(m.CO2_total_tons)),
    })

def solve_pf(df_full: pd.DataFrame):
    m = build_pf_model(df_full)
    opt = SolverFactory(SOLVER_NAME)
    opt.solve(m, tee=SOLVER_TEE)
    pf_design = _extract_pf_design(m)
    # JSON persistieren (für RH_ONLY)
    os.makedirs(EXPORT_BASE_DIR, exist_ok=True)
    with open(PF_DESIGN_JSON, "w", encoding="utf-8") as f:
        json.dump(pf_design, f, indent=2)
    ts = _timeseries_from_model(m, df_full.index)
    ts["strompreis_EUR_MWh"] = df_full["strompreis_EUR_MWh"].values
    costs = _summary_costs(m, ts)
    return ts, costs, pf_design

def run_rolling_horizon(df_full: pd.DataFrame, fixed_design: dict | None = None):
    dt = DT_H
    Ttot = len(df_full)
    W = min(int(round(HEAT_HORIZON_HOURS / dt)), Ttot)
    S = int(round(STEP_HOURS / dt))
    _require(W > 0 and S > 0, "Ungültige Fenster-/Schrittgröße.")
    starts = list(range(0, Ttot - W + 1, S))
    if starts[-1] != Ttot - W:
        starts.append(Ttot - W)

    out_cols = [
        "P_buy","P_sell",
        *(f"HP_Q_{i}" for i in HP_IDS), "HP_Q_total",
        "HKW_P","GTOST_P","P2H_P","BMHKW_P","HWS_P","HWW_P","AVA_P",
        "STO_level","STO_charge","STO_discharge","Q_dump",
        "strompreis_EUR_MWh"
    ]
    out = pd.DataFrame(index=df_full.index, columns=out_cols, dtype=float)

    soc_next = 0.0
    opt = SolverFactory(SOLVER_NAME)

    for s0 in starts:
        s1 = s0 + W
        dfW = df_full.iloc[s0:s1].copy()
        m = build_rh_model(dfW, fixed_design=fixed_design)
        m.SOC_init.set_value(float(soc_next))
        _apply_terminal_policy(m, RH_TERMINAL_POLICY)

        opt.solve(m, tee=SOLVER_TEE)

        idxW = dfW.index
        tsW = _timeseries_from_model(m, idxW)
        tsW["strompreis_EUR_MWh"] = dfW["strompreis_EUR_MWh"].values

        commit_len = S if (s0 != starts[-1]) else len(idxW)
        cs = slice(0, commit_len)
        out.loc[idxW[cs], out_cols[:-1]] = tsW.iloc[cs][out_cols[:-1]].values
        out.loc[idxW[cs], "strompreis_EUR_MWh"] = tsW.iloc[cs]["strompreis_EUR_MWh"].values

        # Übergangszustand
        soc_next = float(tsW["STO_level"].iloc[-1]) if len(tsW) else 0.0

    # „Kosten“ aus RH: aggregiert ohne erneutes Lösen → approximativ über m? RH hat viele Fenster.
    # Wir geben eine kompakte Summary zurück (Peak/Volumes); OBJ über Fenster würde Double-Counting riskieren.
    rh_costs = pd.Series({
        "P_buy_peak_MW": float(out["P_buy"].max(skipna=True)),
        "P_buy_sum_MWh": float((out["P_buy"] * DT_H).sum(skipna=True)),
        "P_sell_sum_MWh": float((out["P_sell"] * DT_H).sum(skipna=True)),
        "HP_total_heat_MWh": float((out["HP_Q_total"] * DT_H).sum(skipna=True)),
    }, name="rh_costs")
    return out, rh_costs

def _write_scenario_workbook(meta: Dict[str, Any],
                             pf_ts: Optional[pd.DataFrame]=None,
                             pf_costs: Optional[pd.Series]=None,
                             pf_design: Optional[Dict[str, Any]]=None,
                             rh_ts: Optional[pd.DataFrame]=None,
                             rh_costs: Optional[pd.Series]=None) -> str:
    os.makedirs(EXPORT_BASE_DIR, exist_ok=True)
    with pd.ExcelWriter(SCENARIO_XLSX, engine="openpyxl") as xw:
        pd.DataFrame(list(meta.items()), columns=["key","value"]).to_excel(xw, "meta", index=False)
        if pf_ts is not None: _excel_safe_df(pf_ts).to_excel(xw, "pf_timeseries")
        if pf_costs is not None: pf_costs.to_frame("value").to_excel(xw, "pf_costs")
        if pf_design is not None:
            pd.DataFrame(list(pf_design.items()), columns=["key","value"]).to_excel(xw, "pf_design", index=False)
        if rh_ts is not None: _excel_safe_df(rh_ts).to_excel(xw, "rh_timeseries")
        if rh_costs is not None: rh_costs.to_frame("value").to_excel(xw, "rh_costs")
    return SCENARIO_XLSX

def run_full_exports_after_runs_scenario(
    scenario_title: str,
    run_mode: str,
    extra_tag: str | None,
    pf_ts: Optional[pd.DataFrame],
    pf_costs: Optional[pd.Series],
    pf_design: Optional[Dict[str, Any]],
    rh_ts: Optional[pd.DataFrame],
    rh_costs: Optional[pd.Series]
):
    stamp = time.strftime("%Y%m%d_%H%M%S")
    safe_title = _slug(scenario_title)
    parts = [safe_title, run_mode, (extra_tag or "").strip()]
    tag = "_".join([p for p in parts if p])

    scen_dir = os.path.join(EXPORT_BASE_DIR, f"{stamp}_{tag}")
    os.makedirs(scen_dir, exist_ok=True)

    meta = {
        "scenario_title": scenario_title,
        "run_mode": run_mode,
        "year_target": YEAR_TARGET,
        "dt_h": DT_H,
        "heat_horizon_hours": HEAT_HORIZON_HOURS,
        "step_hours": STEP_HOURS,
        "terminal_policy": RH_TERMINAL_POLICY,
        "include_demand_charge_in_rh": FLAGS["demand_in_rh"],
        "include_invest_costs_in_rh": FLAGS["capex_in_rh"],
        "include_install_costs_in_rh": FLAGS["install_in_rh"],
        "include_gridcost_in_energy": FLAGS["gridcost_in_energy"],
        "include_co2_cost_in_objective": FLAGS["co2_in_obj"],
        "co2_price_eur_per_t": CO2_PRICE_EUR_PER_T,
        "solver": SOLVER_NAME,
        "input_xlsx": INPUT_XLSX,
        "timestamp": stamp,
        "extra_tag": extra_tag or "",
    }

    # Schreibe das zentrale Workbook in den Szenario-Ordner
    scenario_xlsx_path = os.path.join(scen_dir, os.path.basename(SCENARIO_XLSX))
    _ = _write_scenario_workbook(meta, pf_ts, pf_costs, pf_design, rh_ts, rh_costs)
    shutil.copy2(SCENARIO_XLSX, scenario_xlsx_path)

    # Kopiere das PF-Design-JSON, falls vorhanden
    copied = {"scenario_xlsx": scenario_xlsx_path}
    if os.path.exists(PF_DESIGN_JSON):
        dst = os.path.join(scen_dir, os.path.basename(PF_DESIGN_JSON))
        shutil.copy2(PF_DESIGN_JSON, dst)
        copied["pf_design_json"] = dst

    print(f"[SCENARIO] Zusammenfassung: {scenario_xlsx_path}")
    return {"dir": scen_dir, "scenario_dir": scen_dir, "files": copied, "scenario_xlsx": scenario_xlsx_path}


# ==============================
# 5) Orchestrierung
# ==============================
def run_all(extra_tag: str | None = None):
    df = load_input_excel(
        INPUT_XLSX,
        sheet_name=INPUT_SHEET,
        year_target=YEAR_TARGET,
        ambiguous_policy="first",
        gap_strategy="ffill"
    )

    pf_ts = pf_costs = rh_ts = rh_costs = pf_design = None

    if RUN_MODE in ("PF_ONLY","PF_THEN_RH"):
        pf_ts, pf_costs, pf_design = solve_pf(df)

    if RUN_MODE in ("RH_ONLY","PF_THEN_RH"):
        if pf_design is None:
            try:
                with open(PF_DESIGN_JSON, "r", encoding="utf-8") as f:
                    pf_design = json.load(f)
            except FileNotFoundError:
                pf_design = None
        rh_ts, rh_costs = run_rolling_horizon(df, fixed_design=pf_design)

    return run_full_exports_after_runs_scenario(
        scenario_title=SCENARIO_TITLE,
        run_mode=RUN_MODE,
        extra_tag=extra_tag,
        pf_ts=pf_ts,
        pf_costs=pf_costs,
        pf_design=pf_design,
        rh_ts=rh_ts,
        rh_costs=rh_costs
    )

# ==============================
# 6) Main (Notebook-Ausführung)
# ==============================
res = None
res = run_all(extra_tag='Notebook')
print('\nFertig. Szenario-Zusammenfassung:')
from pprint import pprint; pprint(res)

if isinstance(res, dict) and ('dir' in res or 'scenario_dir' in res):
    print('\nSzenario-Ordner:', res.get('dir', res.get('scenario_dir')))
else:
    print('Hinweis: res ist kein Dict oder enthält keinen Key "dir"/"scenario_dir".')


RUN_MODE       = PF_ONLY
SCENARIO_TITLE = HP_v3_CO2_100
INPUT          = Import_Data.xlsx
EXPORTS        = exports
[Loader] Entferne 2 doppelte Zeitstempel (behalte erste Vorkommen).
[Loader] Es fehlen 2 Zeitschritte (soll: 8760, ist: 8758). gap_strategy='ffill' → fülle Lücken.
[LOAD] Import_Data.xlsx → 8760 Std (UTC-naiv) von 2023-01-01 00:00:00 bis 2023-12-31 23:00:00, dt=1.0h
Read LP format model from file C:\Users\LKR\AppData\Local\Temp\4\tmp97t2s49t.pyomo.lp
Reading time = 1.25 seconds
x1: 315373 rows, 271572 columns, 858503 nonzeros
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows Server 2022.0 (20348.2))

CPU model: Intel(R) Xeon(R) Gold 5220 CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 34 physical cores, 66 logical processors, using up to 32 threads

Academic license 2602149 - for non-commercial use only - registered to lu___@eep.uni-stuttgart.de
Optimize a model with 315373 rows, 271572 columns and 858503 nonzeros
Model fingerprint: 0xbe